# 🧬 Notebook 0 (Optional): Build a Dataset From Any ChIP-seq Peaks File + Reference Genome

**How to read this notebook:**
| Marker | Meaning |
|---|---|
| 🔒 DO NOT EDIT | Infrastructure — just run it |
| ✏️ EDIT ME | A variable is meant to be changed — edit it, then re-run |
| 🔲 FILL IN | A line is intentionally incomplete — write the missing code |

**Why this notebook exists:** Notebooks 1-3 use a fixed dataset at
`~/ctcf_k562_example/`. This notebook shows how that dataset gets BUILT from
two raw inputs — a peaks file and a reference genome — so you can swap in a
different transcription factor, cell type, or species.

**Roadmap**
1. What peaks files and reference genomes actually contain
2. Environment setup
3. 🎯 Configure your dataset (the one section you'll usually edit)
4. Load the reference genome
5. Load and clean the peaks file
6. Extract positive (binding) sequences — **includes an important data-quality fix**
7. Generate background (negative) sequences
8. Write out seqs.txt / labels.txt + a config file
9. Run a full dataset sanity-check suite
10. Point Notebooks 1-3 at your new dataset

## 🎯 SECTION 1: Peaks Files and Reference Genomes, Explained

**A PEAKS FILE** (BED / narrowPeak format) is a plain text table telling us
WHERE in the genome a protein was experimentally found bound to DNA:

    chrom   start     end       name   score  strand  signal  pval  qval  summit
    chr1    1000000   1000300   peak1  500    .       12.4    3.2   2.1   150

We only need the first 3 columns: `chrom`, `start`, `end`.

**A REFERENCE GENOME FASTA file** is the actual DNA sequence for an
organism, organized by chromosome:

    >chr1
    NNNNNNNNNNACGTGCATGCATGC...
    >chr2
    ACGTTTGCATGCATGCATGCATGC...

**🔑 Key Terms**
- **Peak** — one experimentally-confirmed binding location
- **BED format** — a simple tab-separated interval format (chrom, start, end, ...)
- **FASTA format** — the standard text format for DNA/protein sequences
- **N** — a placeholder character meaning "this position couldn't be sequenced confidently" (common in centromeres, telomeres, repetitive regions)

**💡 So what?** Given a peaks file + a reference genome, we can look up the
ACTUAL DNA LETTERS at every peak location. Those become our POSITIVE
examples (label=1). We also need NEGATIVE examples (label=0) — DNA windows
where the protein was NOT found bound.

In [1]:
# 🔒 DO NOT EDIT
print("=" * 65)
print("  What We Need to Build a Dataset")
print("=" * 65)
print()
print("  Peaks file (.bed / .bed.gz / narrowPeak) -> WHERE binding happens")
print("  Reference genome (.fa / .fa.gz)          -> WHAT the DNA says there")
print()
print("  Output: seqs.txt (one DNA sequence per line)")
print("          labels.txt (1 = binding site, 0 = background)")

  What We Need to Build a Dataset

  Peaks file (.bed / .bed.gz / narrowPeak) -> WHERE binding happens
  Reference genome (.fa / .fa.gz)          -> WHAT the DNA says there

  Output: seqs.txt (one DNA sequence per line)
          labels.txt (1 = binding site, 0 = background)


## SECTION 1b: Where to Get These Files📥 Public Sources (Reference Only)

| Resource | Where | Notes |
|---|---|---|
| Peaks files | ENCODE project (encodeproject.org) | Search by transcription factor + cell line, download the "narrowPeak" file |
| Reference genome | UCSC Genome Browser (hgdownload.soe.ucsc.edu/goldenPath) | Pick a species/assembly (e.g. hg38 for human, mm10 for mouse) |

⚠️ Download these to disk BEFORE running this notebook — HPC compute nodes
typically have no internet access.

## 🔧 Setup
This pipeline needs two genomics-specific libraries:
- **pyfaidx** — fast random access into large FASTA files, without loading
  the whole genome into memory
- **pybedtools** — a Python wrapper around the `bedtools` command-line tool,
  for interval math (subtracting, overlapping regions)

Nothing to edit here — just run it and confirm both libraries load and the
`bedtools` binary is found.

In [3]:
#Description: Sets the PATH so the bedtools binary can be found, imports the genomics libraries, and 
#runs one tiny test operation to confirm bedtools itself is reachable — 
#this fails loudly with a clear message if bedtools isn't installed/loaded.
# 🔒 DO NOT EDIT
import os
os.environ["PATH"] = ("/global/cfs/cdirs/m4388/envs/dna-llm/bin:" + os.environ.get("PATH", ""))

import pybedtools
pybedtools.helpers.set_bedtools_path("/global/cfs/cdirs/m4388/envs/dna-llm/bin")

import gzip
import shutil
import random
import json
import pathlib
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    from pyfaidx import Fasta
except ImportError as e:
    raise ImportError(
        "Missing genomics libraries. Install with:\n"
        "  pip install pyfaidx pybedtools pandas tqdm\n"
        "and make sure the 'bedtools' binary is available "
        "(module load bedtools, or conda install -c bioconda bedtools)"
    ) from e

# Confirm the bedtools binary itself is reachable
try:
    pybedtools.BedTool.from_dataframe(pd.DataFrame([["chr1", 0, 10]])).sort()
    print("✅ bedtools binary found and working.")
except Exception as e:
    print("⚠️  bedtools binary not found. Run 'module load bedtools' on NERSC,")
    print("   or 'conda install -c bioconda bedtools' in your environment.")
    print(f"   Details: {e}")

✅ bedtools binary found and working.


## SECTION 3🎯 YOUR TASK: Configure Your Dataset

This is the ONLY section you need to edit to build a dataset for a
DIFFERENT transcription factor, cell type, or species.

| Variable | Meaning |
|---|---|
| `DATASET_NAME` | Short label used to name the output folder |
| `PEAKS_FILE` | Full path to your local peaks `.bed.gz` file |
| `GENOME_FILE` | Full path to your local reference genome `.fa`/`.fa.gz` file |
| `WINDOW_SIZE` | Length (bp) of each DNA window |
| `RANDOM_NEG_MULTIPLIER` | How many background sequences to draw per positive |
| `SEED` | Random seed — keep fixed for reproducibility |

In [4]:
#Description: Sets all the configuration variables for this dataset-building run and creates the output directory 
# everything downstream reads from these variables.

# ✏️ EDIT ME
DATASET_NAME = "ctcf_k562"                                    # rename for your TF/dataset
PEAKS_FILE   = os.path.expanduser("/global/cfs/cdirs/m4388/projects/project7/data/peaks/ENCFF356LFX.bed.gz")
GENOME_FILE  = os.path.expanduser("/global/cfs/cdirs/m4388/projects/project7/data/genome/hg38.fa.gz")

WINDOW_SIZE           = 200   # try 200 (default), 100, or 500
RANDOM_NEG_MULTIPLIER = 1     # try 1 (balanced) or 2 (more background than binding)
SEED                  = 42    # keep fixed for reproducibility!

# 🔒 DO NOT EDIT — canonical chromosomes to keep (avoids alt contigs / scaffolds)
CANONICAL_CHROMS = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}

PROJECT_DIR = Path(os.path.expanduser("~"))
OUTPUT_DIR  = PROJECT_DIR / f"{DATASET_NAME}_example"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("🔧 Your dataset configuration:")
print(f"   DATASET_NAME           = {DATASET_NAME}")
print(f"   PEAKS_FILE             = {PEAKS_FILE}")
print(f"   GENOME_FILE             = {GENOME_FILE}")
print(f"   WINDOW_SIZE             = {WINDOW_SIZE}")
print(f"   RANDOM_NEG_MULTIPLIER   = {RANDOM_NEG_MULTIPLIER}")
print(f"   Output directory        = {OUTPUT_DIR}")

🔧 Your dataset configuration:
   DATASET_NAME           = ctcf_k562
   PEAKS_FILE             = /global/cfs/cdirs/m4388/projects/project7/data/peaks/ENCFF356LFX.bed.gz
   GENOME_FILE             = /global/cfs/cdirs/m4388/projects/project7/data/genome/hg38.fa.gz
   WINDOW_SIZE             = 200
   RANDOM_NEG_MULTIPLIER   = 1
   Output directory        = /global/homes/j/jrios/ctcf_k562_example


## SECTION 4🎯 Fast Random Access Into a 3-Billion-Letter File

The human genome is ~3.2 billion letters long — far too big to search
naively every time we need a snippet. `pyfaidx.Fasta` builds a small INDEX
file (`.fai`) the first time you open a FASTA, then uses that index to jump
DIRECTLY to any position, without reading the whole file into memory.

**🔑 Key Term**
- **Index (`.fai` file)** — a lookup table mapping "chromosome + position" to a byte offset in the file, enabling instant random access

In [5]:
#Decompresses the genome FASTA if needed (pyfaidx requires an uncompressed file) and loads it via pyfaidx, 
#then prints the first few chromosome names found so we can confirm it loaded correctly.
# 🔒 DO NOT EDIT
def load_reference_fasta(path):
    """Load any reference genome FASTA (gzipped or not) using pyfaidx."""
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Reference genome not found: {path}\n"
            f"Download it before running this notebook (see Section 1b)."
        )

    if path.suffix == ".gz":
        print("🗜️  Decompressing genome FASTA (pyfaidx needs an uncompressed file)...")
        out_path = path.with_suffix("")
        if not out_path.exists():
            with gzip.open(path, "rb") as f_in, open(out_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        path = out_path

    print(f"📄 Loading reference genome: {path}")
    fasta = Fasta(str(path), rebuild=False)
    print(f"✅ Loaded {len(fasta.keys())} sequences (chromosomes/contigs).")
    return fasta

fasta = load_reference_fasta(GENOME_FILE)

print()
print("First 5 sequence names found in this genome file:")
for name in list(fasta.keys())[:5]:
    print(f"   {name}  (length={len(fasta[name]):,} bp)")

🗜️  Decompressing genome FASTA (pyfaidx needs an uncompressed file)...
📄 Loading reference genome: /global/cfs/cdirs/m4388/projects/project7/data/genome/hg38.fa
✅ Loaded 455 sequences (chromosomes/contigs).

First 5 sequence names found in this genome file:
   chr1  (length=248,956,422 bp)
   chr10  (length=133,797,422 bp)
   chr11  (length=135,086,622 bp)
   chr11_KI270721v1_random  (length=100,316 bp)
   chr12  (length=133,275,309 bp)


## SECTION 5🎯 Loading and Cleaning the Peaks File

We do THREE things here:
1. **Filter to canonical chromosomes** — genome assemblies also include
   small "alt contigs" and unplaced scaffolds that duplicate or fragment
   real sequence; excluding them keeps sampling simple.
2. **Keep the ORIGINAL peak intervals** — used later so background
   sequences don't accidentally overlap a real binding site.
3. **Build FIXED-WIDTH windows centered on each peak** — real ChIP-seq
   peaks vary in width, but our classifier needs every input to be the
   SAME length, so we center a fixed window on each peak's midpoint.

**🔑 Key Term**
- **Peak midpoint / summit** — the center of a binding region, used as the anchor point for building a fixed-size window

In [7]:
#Reads the peaks BED file, filters to canonical chromosomes only, 
#and builds a fixed-width window centered on each peak's midpoint — printing how many peaks survive each filtering step.

# 🔒 DO NOT EDIT
def load_peak_bed(peaks_path, window_size, canonical_chroms):
    print(f"📄 Loading peaks file: {peaks_path}")

    df = pd.read_csv(
        peaks_path, sep="\t", header=None, compression="gzip",
        usecols=[0, 1, 2], names=["chrom", "start", "end"],
    )
    print(f"   Raw peak count            : {len(df):,}")

    df = df[df["chrom"].isin(canonical_chroms)].copy()
    print(f"   After canonical filtering : {len(df):,}")

    # Keep original intervals for background masking (Section 7)
    peak_bt_original = pybedtools.BedTool.from_dataframe(df[["chrom", "start", "end"]])

    # Build fixed-width windows centered on each peak
    half = window_size // 2
    df["center"] = ((df["start"] + df["end"]) // 2).astype(int)
    df["new_start"] = (df["center"] - half).clip(lower=0)
    df["new_end"] = df["new_start"] + window_size

    peak_bt_windows = pybedtools.BedTool.from_dataframe(df[["chrom", "new_start", "new_end"]])

    print(f"✅ Built {len(peak_bt_windows)} fixed-width ({window_size} bp) positive windows.")
    return peak_bt_original, peak_bt_windows

peak_bt_original, peak_bt_windows = load_peak_bed(PEAKS_FILE, WINDOW_SIZE, CANONICAL_CHROMS)

print()
print("First 3 positive windows:")
for iv in list(peak_bt_windows)[:3]:
    print(f"   {iv.chrom}:{iv.start}-{iv.end}")

📄 Loading peaks file: /global/cfs/cdirs/m4388/projects/project7/data/peaks/ENCFF356LFX.bed.gz
   Raw peak count            : 910
   After canonical filtering : 910
✅ Built 910 fixed-width (200 bp) positive windows.

First 3 positive windows:
   chr1:631903-632103
   chr1:5850229-5850429
   chr1:8909712-8909912


## SECTION 6🎯  Looking Up the Actual DNA Letters — and a Data-Quality Fix

For every fixed-width window, we look up the actual DNA letters from the
reference genome. These become our label=1 examples.

**⚠️ Important fix applied in this version:**
Some genome regions couldn't be sequenced confidently and are marked with
the letter **N** instead of A/C/G/T. If we silently kept peak windows
containing N's, and separately (Section 7) DROPPED any background window
containing N's, the two classes would be cleaned inconsistently — a model
could then learn to associate "contains an N" with "is a binding site,"
which is a data ARTIFACT, not real biology.

**The fix:** we now DROP any positive window that contains an N, exactly
matching the rule already used for background windows in Section 7. This
does cost us a small number of real peaks — but it keeps both classes
honest and comparable, which matters far more than dataset size here.

**🔑 Key Term**
- **Data leakage / shortcut artifact** — when a model learns to exploit an
  accidental pattern in how the dataset was built, instead of the real
  signal you actually want it to learn

In [9]:
#Extracts the DNA sequence for each positive window from the reference genome, 
#DROPPING any window that contains an N — this matches the filtering already applied to background windows, and 
#the cell reports exactly how many peaks were dropped for transparency.

# 🔒 DO NOT EDIT
def extract_sequences(bed, fasta):
    """Extract DNA sequences for each window, dropping any that contain 'N'.

    Dropping N-containing windows here (instead of keeping them) keeps this
    class treated IDENTICALLY to the background class in Section 7, which
    also excludes N-containing windows. See the markdown cell above for why
    this matters.
    """
    seqs = []
    n_dropped = 0
    for iv in tqdm(bed, desc="Extracting positive sequences"):
        seq = fasta[iv.chrom][iv.start:iv.end].seq.upper()
        if "N" in seq:
            n_dropped += 1
            continue
        seqs.append(seq)

    print(f"\n✅ Extracted {len(seqs):,} positive sequences.")
    print(f"⚠️  Dropped {n_dropped:,} peak window(s) containing 'N' "
          f"(kept for symmetry with the background filtering in Section 7).")
    return seqs

positive_seqs = extract_sequences(peak_bt_windows, fasta)

print(f"   Example: {positive_seqs[0][:60]}...")

Extracting positive sequences: 100%|██████████| 910/910 [00:03<00:00, 281.47it/s]


✅ Extracted 907 positive sequences.
⚠️  Dropped 3 peak window(s) containing 'N' (kept for symmetry with the background filtering in Section 7).
   Example: TGATATCAATTGGCTTCCTAGGGTTTATCGTGTGAGCACACCATATATTTACAGTAGGAA...


## SECTION 7🎯 Concept: Building a Fair Contrast Set

We need DNA windows where the protein was NOT bound, so the model has
something to contrast against. Steps:
1. Build a BED file covering the ENTIRE genome.
2. SUBTRACT the original peak intervals — leaving only regions with NO
   known binding.
3. Randomly sample fixed-width windows from those leftover regions until we
   have as many background sequences as positive sequences.
4. Skip any window containing "N" (same rule now used for positives too,
   after the Section 6 fix).

**💡 So what?** This masking step matters a lot — if we sampled background
randomly WITHOUT subtracting real peaks first, we could accidentally label
a true binding site as "background," confusing the model.

**🔑 Key Term**
- **Interval subtraction** — computing "everything in set A that does NOT overlap set B" (here: genome minus known peaks)

In [10]:
#Builds a genome-wide BED file, subtracts the known peak regions from it, then randomly samples fixed-width windows from the remaining "safe" regions, 
#rejecting any window containing an N, until enough background sequences are collected.
# 🔒 DO NOT EDIT
def generate_background(peak_bt_original, fasta, n_needed, window_size, seed, output_dir):
    random.seed(seed)

    chrom_lengths = {chrom: len(fasta[chrom]) for chrom in fasta.keys()}

    genome_bed_path = output_dir / "genome_intervals.bed"
    with genome_bed_path.open("w") as gf:
        for chrom, length in chrom_lengths.items():
            gf.write(f"{chrom}\t0\t{length}\n")

    genome_bt = pybedtools.BedTool(str(genome_bed_path))
    nonpeak_bt = genome_bt.subtract(peak_bt_original, A=True)
    nonpeak_intervals = list(nonpeak_bt)

    if not nonpeak_intervals:
        raise RuntimeError("No non-peak regions found after subtracting peaks from the genome.")

    def draw_window(iv):
        max_start = iv.end - window_size
        if max_start <= iv.start:
            return None
        s = random.randint(iv.start, max_start)
        return pybedtools.Interval(iv.chrom, s, s + window_size)

    background = []
    attempts, max_attempts = 0, n_needed * 500
    pbar = tqdm(total=n_needed, desc="Sampling background")

    while len(background) < n_needed and attempts < max_attempts:
        attempts += 1
        iv = random.choice(nonpeak_intervals)
        win = draw_window(iv)
        if win is None:
            continue
        seq = fasta[win.chrom][win.start:win.end].seq.upper()
        if "N" in seq:
            continue
        background.append(seq)
        pbar.update(1)
    pbar.close()

    print(f"✅ Generated {len(background):,} background windows "
          f"({attempts:,} attempts, {attempts - len(background):,} rejected).")
    return background

n_background_needed = len(positive_seqs) * RANDOM_NEG_MULTIPLIER
background_seqs = generate_background(
    peak_bt_original, fasta, n_background_needed, WINDOW_SIZE, SEED, OUTPUT_DIR
)

Sampling background: 100%|██████████| 907/907 [00:12<00:00, 71.12it/s]

✅ Generated 907 background windows (926 attempts, 19 rejected).


## SECTION 8🎯  Reproducible Config Alongside Data

This matches EXACTLY the format Notebooks 1-3 expect:
- `seqs.txt` — one DNA sequence per line
- `labels.txt` — one label per line (1 = binding, 0 = background), in the SAME order

We also save `dataset_config.json` recording exactly how this dataset was
built, including how many N-containing peaks were dropped — the same
"reproducible config" habit used throughout this bootcamp.

In [12]:
#Writes the final positive and background sequences to seqs.txt/labels.txt in matching order, 
#then saves a config JSON recording every setting and count used to build this dataset.
# 🔒 DO NOT EDIT
def write_dataset(pos, neg, output_dir):
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    with (output_dir / "seqs.txt").open("w") as sf, (output_dir / "labels.txt").open("w") as lf:
        for s in pos:
            sf.write(s + "\n")
            lf.write("1\n")
        for s in neg:
            sf.write(s + "\n")
            lf.write("0\n")

    print(f"📁 Dataset written to {output_dir}")
    print(f"   seqs.txt   : {len(pos) + len(neg):,} lines")
    print(f"   labels.txt : {len(pos) + len(neg):,} lines")

write_dataset(positive_seqs, background_seqs, OUTPUT_DIR)

config_record = {
    "dataset_name": DATASET_NAME,
    "peaks_file": str(PEAKS_FILE),
    "genome_file": str(GENOME_FILE),
    "window_size": WINDOW_SIZE,
    "random_neg_multiplier": RANDOM_NEG_MULTIPLIER,
    "seed": SEED,
    "n_positive": len(positive_seqs),
    "n_background": len(background_seqs),
    "n_containing_windows_excluded_from_both_classes": True,
}
with open(OUTPUT_DIR / "dataset_config.json", "w") as f:
    json.dump(config_record, f, indent=2)

print(f"✅ Config saved to {OUTPUT_DIR / 'dataset_config.json'}")

📁 Dataset written to /global/homes/j/jrios/ctcf_k562_example
   seqs.txt   : 1,814 lines
   labels.txt : 1,814 lines
✅ Config saved to /global/homes/j/jrios/ctcf_k562_example/dataset_config.json


## SECTION 9🎯  Confirming the Fix Worked, and Auditing the Full Dataset

Before handing this dataset to Notebooks 1-3, we run the SAME six-check
suite used in Notebook 1, so both notebooks give you a consistent way to
audit any dataset:

| # | Check | Catches |
|---|---|---|
| 1 | Class balance | Skewed classes needing re-weighting or resampling |
| 2 | Sequence length consistency | Bugs in how windows were built |
| 3 | Character validity & N-content (by label) | Confirms the Section 6 fix worked — both classes should now show 0% N-content |
| 4 | Duplicate sequences | Risk of the same sequence in both train and validation later |
| 5 | GC content distribution | Expected real biological signal (CTCF sites are GC-rich) |
| 6 | Per-position nucleotide composition | Confirms binding signal is centered on the window, as expected from how peaks were built |

In [14]:
#Reloads the written seqs.txt/labels.txt from disk (to test the actual files, not just in-memory variables) and
#runs Checks 1-2: class balance and sequence length consistency.
# 🔒 DO NOT EDIT
with open(OUTPUT_DIR / "seqs.txt") as f:
    check_seqs = [line.strip() for line in f if line.strip()]
with open(OUTPUT_DIR / "labels.txt") as f:
    check_labels = [int(line.strip()) for line in f if line.strip()]

labels_arr = np.array(check_labels)
seq_lengths = np.array([len(s) for s in check_seqs])

print("CHECK 1: Class Balance")
n_pos = int(labels_arr.sum())
n_neg = len(labels_arr) - n_pos
print(f"  Binding (1)   : {n_pos:,}")
print(f"  Background (0): {n_neg:,}")
print(f"  Ratio         : {n_pos / n_neg:.2f} : 1")
print()

print("CHECK 2: Sequence Length Consistency")
expected_len = int(np.median(seq_lengths))
n_wrong_length = int((seq_lengths != expected_len).sum())
print(f"  Expected length (WINDOW_SIZE): {expected_len} bp")
print(f"  Sequences with a DIFFERENT length: {n_wrong_length}")
if n_wrong_length > 0:
    print("  ⚠️  Investigate before continuing!")
else:
    print("  ✅ All sequences are the same length.")

CHECK 1: Class Balance
  Binding (1)   : 907
  Background (0): 907
  Ratio         : 1.00 : 1

CHECK 2: Sequence Length Consistency
  Expected length (WINDOW_SIZE): 200 bp
  Sequences with a DIFFERENT length: 0
  ✅ All sequences are the same length.


In [15]:
#Checks character validity and N-content by label 
import collections

all_chars_seen = set("".join(check_seqs))
unexpected_chars = all_chars_seen - set("ACGT")   # note: no "N" expected anymore!

print("CHECK 3: Character Validity & N-Content (post-fix)")
print(f"  All characters seen : {sorted(all_chars_seen)}")
print(f"  Unexpected characters: {unexpected_chars if unexpected_chars else 'None'}")
print()

n_count_by_label = {0: 0, 1: 0}
total_by_label = {0: 0, 1: 0}
for seq, label in zip(check_seqs, check_labels):
    total_by_label[label] += 1
    if "N" in seq:
        n_count_by_label[label] += 1

print("  Sequences containing at least one 'N':")
for label in [0, 1]:
    pct = 100 * n_count_by_label[label] / total_by_label[label]
    name = "Background" if label == 0 else "Binding"
    print(f"    {name:<10} (label={label}): {n_count_by_label[label]} / {total_by_label[label]} ({pct:.1f}%)")

if n_count_by_label[0] == 0 and n_count_by_label[1] == 0:
    print()
    print("  ✅ Both classes are N-free — the Section 6 fix worked as intended.")
else:
    print()
    print("  ⚠️  N's still detected — double check Section 6 ran the patched version.")

CHECK 3: Character Validity & N-Content (post-fix)
  All characters seen : ['A', 'C', 'G', 'T']
  Unexpected characters: None

  Sequences containing at least one 'N':
    Background (label=0): 0 / 907 (0.0%)
    Binding    (label=1): 0 / 907 (0.0%)

  ✅ Both classes are N-free — the Section 6 fix worked as intended.


In [19]:
#Counts exact duplicate sequences across the whole dataset, which matters 
#because duplicates split across train/validation later could leak information between the two sets.
# 🔒 DO NOT EDIT
seq_counts = collections.Counter(check_seqs)
duplicates = {seq: c for seq, c in seq_counts.items() if c > 1}

print("CHECK 4: Duplicate Sequences")
print(f"  Total sequences  : {len(check_seqs):,}")
print(f"  Unique sequences : {len(seq_counts):,}")
print(f"  Sequences appearing MORE than once: {len(duplicates)}")

if duplicates:
    print("  ⚠️  Duplicates found — worth investigating before splitting train/val.")
else:
    print("  ✅ No exact duplicate sequences found.")

CHECK 4: Duplicate Sequences
  Total sequences  : 1,814
  Unique sequences : 1,813
  Sequences appearing MORE than once: 1
  ⚠️  Duplicates found — worth investigating before splitting train/val.


## 🔍 Check 4b: Investigate the Specific Duplicate(s)

Check 4 told us THAT a duplicate exists, but not WHERE it came from or
whether it's a harmless coincidence or a real label conflict. Let's find out.

In [18]:
# 🔒 DO NOT EDIT
duplicate_seqs = [seq for seq, c in seq_counts.items() if c > 1]

for dup_seq in duplicate_seqs:
    indices = [i for i, s in enumerate(check_seqs) if s == dup_seq]
    dup_labels = [check_labels[i] for i in indices]

    print(f"Duplicate sequence: {dup_seq[:60]}...")
    print(f"  Appears at indices : {indices}")
    print(f"  Labels at those indices: {dup_labels}")

    if len(set(dup_labels)) == 1:
        name = "Binding" if dup_labels[0] == 1 else "Background"
        print(f"  ✅ SAME-LABEL duplicate (both '{name}'). Likely a coincidental")
        print(f"     repeat — e.g. two peaks close enough to produce near-identical")
        print(f"     windows, or background sampling hit the same repetitive region twice.")
    else:
        print(f"  🚨 LABEL CONFLICT — the SAME sequence is labeled BOTH binding AND")
        print(f"     background! This must be resolved before training, or the")
        print(f"     model will get contradictory signal for this exact input.")
    print()

Duplicate sequence: ACCCTGGAGGTGCAGTGGTGACTAAGGCACACTTCTGTGTCTTGTGGGGTTCTGAGTCAT...
  Appears at indices : [233, 234]
  Labels at those indices: [1, 1]
  ✅ SAME-LABEL duplicate (both 'Binding'). Likely a coincidental
     repeat — e.g. two peaks close enough to produce near-identical
     windows, or background sampling hit the same repetitive region twice.



## 🎯 Concept: Deduplicating Before We Save

Now that we know what kind of duplicate we're dealing with, let's remove it
BEFORE writing the final `seqs.txt`/`labels.txt` — this way every notebook
downstream (1, 2, 3) automatically gets the cleaned version, and you never
have to remember to re-check this by hand.

**Policy used below:**
- Same-label duplicates → keep only the FIRST occurrence, drop the rest
- Label-conflict duplicates → drop ALL occurrences of that sequence entirely
  (safer than guessing which label is "more correct")

In [20]:
# 🔒 DO NOT EDIT
def deduplicate_dataset(seqs, labels):
    seq_to_labels = collections.defaultdict(list)
    for seq, label in zip(seqs, labels):
        seq_to_labels[seq].append(label)

    kept_seqs, kept_labels = [], []
    n_same_label_dropped = 0
    n_conflict_dropped = 0

    for seq, lbls in seq_to_labels.items():
        if len(set(lbls)) == 1:
            # same-label duplicate(s): keep exactly one copy
            kept_seqs.append(seq)
            kept_labels.append(lbls[0])
            n_same_label_dropped += len(lbls) - 1
        else:
            # label conflict: drop entirely, keep neither
            n_conflict_dropped += len(lbls)

    print(f"Deduplication summary:")
    print(f"  Same-label duplicate copies removed : {n_same_label_dropped}")
    print(f"  Label-conflict sequences removed entirely: {n_conflict_dropped}")
    print(f"  Final sequence count: {len(kept_seqs):,} (was {len(seqs):,})")

    return kept_seqs, kept_labels

# Recombine positive_seqs/background_seqs into one list+label pair, dedupe, then split back
combined_seqs = positive_seqs + background_seqs
combined_labels = [1] * len(positive_seqs) + [0] * len(background_seqs)

deduped_seqs, deduped_labels = deduplicate_dataset(combined_seqs, combined_labels)

positive_seqs   = [s for s, l in zip(deduped_seqs, deduped_labels) if l == 1]
background_seqs = [s for s, l in zip(deduped_seqs, deduped_labels) if l == 0]

print()
print(f"Final positive count   : {len(positive_seqs):,}")
print(f"Final background count : {len(background_seqs):,}")

Deduplication summary:
  Same-label duplicate copies removed : 1
  Label-conflict sequences removed entirely: 0
  Final sequence count: 1,813 (was 1,814)

Final positive count   : 906
Final background count : 907


## ✅ Checkpoint
**🧪 Try it yourself:** Re-run Section 8 (writing seqs.txt/labels.txt) and
then Check 4 in Section 9 again. `Sequences appearing MORE than once` should
now read `0`.

**📝 Note:** deduplication can leave your two classes slightly imbalanced
(e.g. 909 binding vs. 910 background) if the duplicate happened to be a
label conflict. That's fine — Check 1 (class balance) will show you the
exact new ratio, and it's a tiny enough shift not to worry about at this scale.

In [21]:
#overwrites the final positive and background sequences to seqs.txt/labels.txt in matching order, 
#then saves a config JSON recording every setting and count used to build this dataset.
# 🔒 DO NOT EDIT
def write_dataset(pos, neg, output_dir):
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    with (output_dir / "seqs.txt").open("w") as sf, (output_dir / "labels.txt").open("w") as lf:
        for s in pos:
            sf.write(s + "\n")
            lf.write("1\n")
        for s in neg:
            sf.write(s + "\n")
            lf.write("0\n")

    print(f"📁 Dataset written to {output_dir}")
    print(f"   seqs.txt   : {len(pos) + len(neg):,} lines")
    print(f"   labels.txt : {len(pos) + len(neg):,} lines")

write_dataset(positive_seqs, background_seqs, OUTPUT_DIR)

config_record = {
    "dataset_name": DATASET_NAME,
    "peaks_file": str(PEAKS_FILE),
    "genome_file": str(GENOME_FILE),
    "window_size": WINDOW_SIZE,
    "random_neg_multiplier": RANDOM_NEG_MULTIPLIER,
    "seed": SEED,
    "n_positive": len(positive_seqs),
    "n_background": len(background_seqs),
    "n_containing_windows_excluded_from_both_classes": True,
}
with open(OUTPUT_DIR / "dataset_config.json", "w") as f:
    json.dump(config_record, f, indent=2)

print(f"✅ Config saved to {OUTPUT_DIR / 'dataset_config.json'}")

📁 Dataset written to /global/homes/j/jrios/ctcf_k562_example
   seqs.txt   : 1,813 lines
   labels.txt : 1,813 lines
✅ Config saved to /global/homes/j/jrios/ctcf_k562_example/dataset_config.json
